In [1]:
import vrep 
import sys
import time 
import numpy as np
from tank import *
import skfuzzy as fuzz

In [ ]:
'''
* ‘centroid’: Centroid of area 
* ‘bisector’: bisector of area 
* ‘mom’ : mean of maximum 
* ‘som’ : min of maximum 
* ‘lom’ : max of maximum
'''
mode = 'centroid'

In [2]:
# Universal Set

distance_2 = np.arange(0.5, 3.001, 0.001)
distance_3 = np.arange(0.5, 3.001, 0.001)
distance_4 = np.arange(0.5, 3.001, 0.001)
distance_5 = np.arange(0.5, 3.001, 0.001)
turning = np.arange(0,3,1)
breaking = np.arange(0,3,1)

# Membership Functions
distance_2_poor = fuzz.trimf(distance_2,[0.5,0.5,1.75])
distance_2_average = fuzz.trimf(distance_2,[0.5,1.75,3])
distance_2_good = fuzz.trimf(distance_2,[1.75,3,3])

distance_3_poor = fuzz.trimf(distance_3,[0.5,0.5,1.75])
distance_3_average = fuzz.trimf(distance_3,[0.5,1.75,3])
distance_3_good = fuzz.trimf(distance_3,[1.75,3,3])

distance_4_poor = fuzz.trimf(distance_4,[0.5,0.5,1.75])
distance_4_average = fuzz.trimf(distance_4,[0.5,1.75,3])
distance_4_good = fuzz.trimf(distance_4,[1.75,3,3])

distance_5_poor = fuzz.trimf(distance_5,[0.5,0.5,1.75])
distance_5_average = fuzz.trimf(distance_5,[0.5,1.75,3])
distance_5_good = fuzz.trimf(distance_5,[1.75,3,3])

turning_poor = fuzz.trimf(turning,[0,0,1])
turning_average = fuzz.trimf(turning,[0,1,2])
turning_good = fuzz.trimf(turning,[1,2,2])

breaking_poor = fuzz.trimf(breaking,[0,0,1])
breaking_average = fuzz.trimf(breaking,[0,1,2])
breaking_good = fuzz.trimf(breaking,[1,2,2])

# Interpolation of received inputs from the sensors --> fuzz.interp_membership
distance_2_p = fuzz.interp_membership(distance_2,distance_2_poor,1.281520503512389)
distance_2_a = fuzz.interp_membership(distance_2,distance_2_average,1.281520503512389)
distance_2_g = fuzz.interp_membership(distance_2,distance_2_good,1.281520503512389)

distance_5_p = fuzz.interp_membership(distance_5,distance_5_poor,0.9713329154866324)
distance_5_a = fuzz.interp_membership(distance_5,distance_5_average,0.9713329154866324)
distance_5_g = fuzz.interp_membership(distance_5,distance_5_good,0.9713329154866324)

# Rules
        # Turning
turn_rule1 = np.fmin(distance_2_a,distance_5_g)
turn_1 = np.fmin(turn_rule1,turning_good)
turn_rule2 = np.fmin(distance_2_a,distance_5_a)
turn_2 = np.fmin(turn_rule2,turning_average)

        # Breaking
break_rule1 = np.fmin(distance_2_p,distance_5_p)
break_1 = np.fmin(break_rule1,breaking_good)
break_rule2 = np.fmin(distance_2_g,distance_5_g)
break_2 = np.fmin(break_rule2,breaking_poor)

# Aggregate the Membership Functions
aggregated1 = np.fmax(turn_1,turn_2)
aggregated2 = np.fmax(break_1,break_2)

# Defuzzify the result
#turning_force = fuzz.defuzz(turning,aggregated1,'centroid')
breaking_force = fuzz.defuzz(breaking, aggregated2, mode)
breaking_force

1.7071067811865475

In [3]:
def compute_turning_force(dist2, dist5, method):
    
    # Interpolation of received inputs from the sensors --> fuzz.interp_membership
    distance_2_p = fuzz.interp_membership(distance_2,distance_2_poor,dist2)
    distance_2_a = fuzz.interp_membership(distance_2,distance_2_average,dist2)
    distance_2_g = fuzz.interp_membership(distance_2,distance_2_good,dist2)

    distance_5_p = fuzz.interp_membership(distance_5,distance_5_poor,dist5)
    distance_5_a = fuzz.interp_membership(distance_5,distance_5_average,dist5)
    distance_5_g = fuzz.interp_membership(distance_5,distance_5_good,dist5)
    
    # Rules
    # Turning
    turn_rule1 = np.fmin(distance_2_a,distance_5_g)
    turn_1 = np.fmin(turn_rule1,turning_good)
    turn_rule2 = np.fmin(distance_2_a,distance_5_a)
    turn_2 = np.fmin(turn_rule2,turning_average)

    # Aggregate the Membership Functions
    aggregated1 = np.fmax(turn_1,turn_2)

    # Defuzzify the result
    turning_force = fuzz.defuzz(turning,aggregated1, method)
    
    return turning_force

def compute_breaking_force(dist2, dist5, method):
    
    # Interpolation of received inputs from the sensors --> fuzz.interp_membership
    distance_2_p = fuzz.interp_membership(distance_2,distance_2_poor,dist2)
    distance_2_a = fuzz.interp_membership(distance_2,distance_2_average,dist2)
    distance_2_g = fuzz.interp_membership(distance_2,distance_2_good,dist2)

    distance_5_p = fuzz.interp_membership(distance_5,distance_5_poor,dist5)
    distance_5_a = fuzz.interp_membership(distance_5,distance_5_average,dist5)
    distance_5_g = fuzz.interp_membership(distance_5,distance_5_good,dist5)
    
    # Rules

    # Breaking
    break_rule1 = np.fmin(distance_2_p,distance_5_p)
    break_1 = np.fmin(break_rule1,breaking_good)
    break_rule2 = np.fmin(distance_2_g,distance_5_g)
    break_2 = np.fmin(break_rule2,breaking_poor)

    # Aggregate the Membership Functions

    aggregated2 = np.fmax(break_1,break_2)

    # Defuzzify the result
    turning_force = fuzz.defuzz(breaking,aggregated2, method)
    
    return breaking_force

    

In [4]:
def detect_parking_space():
    prev = 5
    while True:
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,proximity_sensors_handles[2],vrep.simx_opmode_buffer)
        dist = np.linalg.norm(detectedPoint)
        if prev != 0 and dist / prev > 2:
            break
        prev = dist
    print('Found empty space')
    print('Getting into position')
    
def start_parking():
    prev = 1
    while True:
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,proximity_sensors_handles[4],vrep.simx_opmode_buffer)
        dist = np.linalg.norm(detectedPoint)
        if prev != 0 and prev / dist > 2:
            break
        prev = dist
    tank.stop()
    print('Start parking')
    tank.turn_left(10)
    time.sleep(1.6*2.5)
    tank.backward(10)

def adjust_parking():
    prev = 5
    while True:
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,proximity_sensors_handles[4],vrep.simx_opmode_buffer)
        dist = np.linalg.norm(detectedPoint)
        if prev != 0 and dist / prev > 2:
            break
        prev = dist
    print('Adjusting')
    tank.turn_left(10)

    while True:
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector = vrep.simxReadProximitySensor(clientID,proximity_sensors_handles[2],vrep.simx_opmode_buffer)
        dist_2 = np.linalg.norm(detectedPoint)
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector = vrep.simxReadProximitySensor(clientID,proximity_sensors_handles[5],vrep.simx_opmode_buffer)
        dist_5 = np.linalg.norm(detectedPoint)

        #breaking_force.input['distance_2'] = dist_2
        #breaking_force.input['distance_5'] = dist_5

        try:
            #breaking_force.compute()
            output = compute_breaking_force(dist_2,dist_5, mode)
            time.sleep(output*0.7)
            break
        except:
            pass

    tank.backward(10)
    
def stop_parking():
    while True:
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,proximity_sensors_handles[2],vrep.simx_opmode_buffer)
        dist_2 = np.linalg.norm(detectedPoint)
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,proximity_sensors_handles[5],vrep.simx_opmode_buffer)
        dist_5 = np.linalg.norm(detectedPoint)

        #breaking_force.input['distance_2'] = dist_2
        #breaking_force.input['distance_5'] = dist_5

        try:
            #breaking_force.compute()
            output = compute_turning_force(dist_2,dist_5, mode)
            print(output)
    
            if output < 1:
                tank.stop()
                break
        except Exception as e:
            pass
            # print(e)
    print('Stopping')
    tank.turn_right(10)
    time.sleep(5)
    tank.stop()

def finish_parking():
    tank.forward(5)

    while True:
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,proximity_sensors_handles[2],vrep.simx_opmode_buffer)
        dist_2 = np.linalg.norm(detectedPoint)
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,proximity_sensors_handles[5],vrep.simx_opmode_buffer)
        dist_5 = np.linalg.norm(detectedPoint)

        #breaking_force.input['distance_2'] = dist_2
        #breaking_force.input['distance_5'] = dist_5

        try:
            #breaking_force.compute()
            output = compute_breaking_force(dist_2,dist_5, mode)
            if output > 1.61:
                tank.stop()
                break
        except:
            pass
    print('Finished')

In [5]:
vrep.simxFinish(-1) # closes all opened connections, in case any prevoius wasnt finished
clientID=vrep.simxStart('127.0.0.1',19997,True,True,5000,5) # start a connection

if clientID!=-1:
    print ("Connected to remote API server")
else:
    print("Not connected to remote API server")
    sys.exit("Could not connect")

#create instance of Tank
tank=Tank(clientID)

proximity_sensors=["EN","ES","NE","NW","SE","SW","WN","WS"]
proximity_sensors_handles=[0]*8

# get handle to proximity sensors
for i in range(len(proximity_sensors)):
    err_code,proximity_sensors_handles[i] = vrep.simxGetObjectHandle(clientID,"Proximity_sensor_"+proximity_sensors[i], vrep.simx_opmode_blocking)

err_code,tank_handle = vrep.simxGetObjectHandle(clientID,'Edgeless', vrep.simx_opmode_blocking)

#read and print values from proximity sensors
#first reading should be done with simx_opmode_streaming, further with simx_opmode_buffer parameter
for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_streaming)
        # print(err_code, detectionState, detectedPoint, detectedObjectHandle, detectedSurfaceNormalVector)

#err_code,tank_handle = vrep.simxGetObjectHandle(clientID,'Edgeless', vrep.simx_opmode_blocking)
#err_code,lin_velo,ang_velo = vrep.simxGetObjectVelocity(clientID,tank_handle, vrep.simx_opmode_blocking)

Connected to remote API server


In [6]:
tank.forward(10)

detect_parking_space()
start_parking()
adjust_parking()
stop_parking()
finish_parking()

rea is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzification!
Total area is zero in defuzzific

KeyboardInterrupt: 